# 07 · Self-Attention 从零实现

> **学习目标**：手撸 single-head 与 multi-head self-attention，结果与 `nn.MultiheadAttention` **逐元素对齐**。理解 mask、scaling、heads 在干嘛。
>
> **预备**：04 + 05 + 06 已过。
>
> **为什么重要**：所有 LLM 的中心都是这个东西。它的细节（mask、scaling、reshape）只有自己手写一次才真懂。

**Attention 公式**：

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import math
torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## 1. 单头 self-attention：从 Q, K, V 出发

**心智模型**：每个 token 都「问问每个其它 token 你跟我相关吗」（得到一组权重，softmax 归一）→ 「按相关度把它们的 V 加权合起来」就是新的表示。

Shape 全程：`(B=batch, L=seq_len, D=d_model)`

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (B, L_q, D)
    K, V: (B, L_k, D)   # 通常 L_q == L_k
    mask: (B, L_q, L_k) or None；mask 为 True 的位置会被 -inf
    返回:
      out: (B, L_q, D)
      attn: (B, L_q, L_k)   注意力权重，可视化用
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)   # (B, L_q, L_k)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    attn = scores.softmax(dim=-1)
    out = attn @ V                                       # (B, L_q, D)
    return out, attn

# Smoke test：输入输出形状
B, L, D = 2, 5, 16
Q = torch.randn(B, L, D)
K = torch.randn(B, L, D)
V = torch.randn(B, L, D)
out, attn = scaled_dot_product_attention(Q, K, V)
print('out shape :', out.shape, '应为', (B, L, D))
print('attn shape:', attn.shape, '应为', (B, L, L))
print('每行 attn 和应为 1:', attn.sum(-1)[0, 0].item())

## 2. 为什么要 `/ sqrt(d_k)`？

`Q @ K^T` 的方差大致是 `d_k`（每个维度独立方差为 1）。维度大了，scores 数量级也大，softmax 一过就接近 one-hot，**梯度消失**。除以 `sqrt(d_k)` 让方差回到 1。

In [ ]:
for d in [4, 64, 1024]:
    q = torch.randn(d); k = torch.randn(d)
    print(f'd={d:4}  Q·K  方差≈{(q @ k).item():+.2f}  (无 scale)')
    print(f'        Q·K/√d 方差≈{(q @ k / math.sqrt(d)).item():+.2f}  (有 scale)')

# 看 softmax 是否被锐化
x_unscaled = torch.tensor([10., 11., 12.])
x_scaled   = x_unscaled / math.sqrt(64)
print('\n无 scale softmax:', x_unscaled.softmax(-1))
print('有 scale softmax:', x_scaled.softmax(-1))

## 3. Causal mask：LLM 不准看未来

**问题**：训练时模型一次见到整句话，但生成时是一个 token 一个 token 出。如果允许位置 t 看到位置 t+1 的信息，那就「作弊」了，推理时学不到。

**解决**：上三角的位置全部 mask 成 `-inf`，softmax 后变 0，**完全不参与加权**。

In [ ]:
L = 5
causal_mask = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1)
print('causal mask (True 处会被屏蔽):\n', causal_mask.int())

Q = torch.randn(1, L, 16)
K = torch.randn(1, L, 16)
V = torch.randn(1, L, 16)
out, attn = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
print('\n带 causal mask 后的 attn (位置 i 只能看到 0..i):')
print(attn[0])

## 4. Multi-Head Attention：从一个房间到 H 个房间

**核心理解**：把 D 维分成 H 份，每份 d_k=D/H 维**独立**做 attention，最后 concat 再过一个 Linear。

**为什么**：让模型可以**同时关注不同模式**（一组头看句法、另一组看语义、再一组看远距离指代）。

Shape 演变（重点理解）：
```
(B, L, D)
  → 3 个 Linear → Q, K, V each (B, L, D)
  → reshape 拆头 → (B, L, H, d_k) → permute → (B, H, L, d_k)
  → scaled_dot_product → (B, H, L, d_k)
  → permute + reshape 合并头 → (B, L, D)
  → 输出 Linear → (B, L, D)
```

In [ ]:
class MyMultiheadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # 4 个权重矩阵：Q / K / V 各一个、输出一个
        self.W_q = nn.Linear(d_model, d_model, bias=True)
        self.W_k = nn.Linear(d_model, d_model, bias=True)
        self.W_v = nn.Linear(d_model, d_model, bias=True)
        self.W_o = nn.Linear(d_model, d_model, bias=True)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x):
        # (B, L, D) -> (B, H, L, d_k)
        B, L, _ = x.shape
        return x.view(B, L, self.n_heads, self.d_k).transpose(1, 2)

    def _merge_heads(self, x):
        # (B, H, L, d_k) -> (B, L, D)
        B, H, L, d_k = x.shape
        return x.transpose(1, 2).contiguous().view(B, L, H * d_k)

    def forward(self, q, k, v, attn_mask=None):
        # q,k,v: (B, L, D)
        Q = self._split_heads(self.W_q(q))     # (B, H, Lq, d_k)
        K = self._split_heads(self.W_k(k))     # (B, H, Lk, d_k)
        V = self._split_heads(self.W_v(v))     # (B, H, Lk, d_k)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)    # (B, H, Lq, Lk)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, float('-inf'))
        attn = scores.softmax(dim=-1)
        attn = self.dropout(attn)

        ctx = attn @ V                                              # (B, H, Lq, d_k)
        out = self.W_o(self._merge_heads(ctx))                       # (B, Lq, D)
        return out, attn

# Smoke test
B, L, D, H = 2, 7, 16, 4
mha = MyMultiheadAttention(D, H)
x = torch.randn(B, L, D)
y, attn = mha(x, x, x)
print('input ', x.shape, '-> output', y.shape, '   (B, L, D 必须保持)')
print('attn shape:', attn.shape, '   (B, H, L, L)')

## 5. 与 `nn.MultiheadAttention` 对齐

把官方实现的权重复制到我们的实现里，看输出是不是逐元素一样。**对齐 = 真懂**；不对齐 = 哪里写错了。

In [ ]:
B, L, D, H = 2, 5, 16, 4
torch.manual_seed(42)
x = torch.randn(B, L, D)

# 官方版（注意 batch_first=True）
off = nn.MultiheadAttention(embed_dim=D, num_heads=H, bias=True, batch_first=True)
off.eval()
y_off, attn_off = off(x, x, x, average_attn_weights=False)

# 我们的实现
mine = MyMultiheadAttention(D, H)
mine.eval()

# 把官方的 in_proj_weight 拆成 W_q, W_k, W_v 给我们
with torch.no_grad():
    Wq, Wk, Wv = off.in_proj_weight.chunk(3, dim=0)      # 每块 (D, D)
    bq, bk, bv = off.in_proj_bias.chunk(3, dim=0)
    mine.W_q.weight.copy_(Wq); mine.W_q.bias.copy_(bq)
    mine.W_k.weight.copy_(Wk); mine.W_k.bias.copy_(bk)
    mine.W_v.weight.copy_(Wv); mine.W_v.bias.copy_(bv)
    mine.W_o.weight.copy_(off.out_proj.weight)
    mine.W_o.bias.copy_(off.out_proj.bias)

y_mine, attn_mine = mine(x, x, x)

diff = (y_off - y_mine).abs().max().item()
print(f'输出最大逐元素差: {diff:.2e}    {"✅ 对齐成功" if diff < 1e-5 else "❌ 不一致，检查 reshape 顺序"}')

## 6. 可视化：模型在「看哪儿」

用一个简单序列，画 attention 权重热图，直观感受「self」的含义。

In [ ]:
import matplotlib.pyplot as plt

tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat']
L = len(tokens)
torch.manual_seed(1)
x = torch.randn(1, L, 16)

mha = MyMultiheadAttention(16, 2)
_, attn = mha(x, x, x)        # attn: (1, H=2, L, L)
attn = attn[0].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for h, ax in enumerate(axes):
    im = ax.imshow(attn[h], cmap='viridis')
    ax.set_xticks(range(L)); ax.set_xticklabels(tokens, rotation=45)
    ax.set_yticks(range(L)); ax.set_yticklabels(tokens)
    ax.set_title(f'Head {h}')
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()
print('行 = query token；列 = 它在「看」哪些 token。颜色越亮权重越大。')
print('（模型未训练，所以是随机初值的 attention，仅作 shape 与 API 演示。）')

## 深入思考

1. **Single-head 不够吗，为什么要 multi-head？**
   - 单头只能关注一种「模式」。多头让模型在**不同子空间**并行做不同关注（句法 vs 语义 vs 远距离指代等）。
2. **`d_k` 选多少合理？**
   - 经验值 32~128。太小表达力不足，太大计算昂贵。GPT-3 175B 用 `d_model=12288, n_heads=96` → `d_k=128`。
3. **cross-attention 和 self-attention 区别？**
   - cross：Q 来自一边（如 decoder），K/V 来自另一边（如 encoder）。Encoder-Decoder Transformer 用，decoder-only LLM 没有。
4. **Attention 的时间复杂度？**
   - `O(L² · d)`。L=10k 时 L² = 100M，再乘 d 与层数与 head 数，**长上下文 attention 是显存大头**。FlashAttention / RingAttention / PagedAttention 都是为了打破这条诅咒。
5. **mask 为啥用 `-inf` 而不是 0？**
   - softmax 之前要让那些位置归一化后概率为 0。`-inf` → `exp(-inf) = 0`，干净利落。直接乘 0 会破坏 softmax 的归一性。

改一改：把 `n_heads` 改成 1，重跑对齐测试，验证 single-head 也对齐。

## 自检 ✅

- [ ] 不看代码，写出 `scaled_dot_product_attention` 的完整 4 步公式。
- [ ] 解释「为什么要 `/ sqrt(d_k)`」。
- [ ] 解释 causal mask 是怎么工作的，画出 5×5 的样子。
- [ ] 默写 multi-head 的 5 步 shape 演变。
- [ ] 给一段 attention 实现代码，能 1 分钟内看出 head 数和 d_model。

## 下一步

→ [`08_qwen_sampling.ipynb`](08_qwen_sampling.ipynb)